In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download ronikdedhia/next-word-prediction

Dataset URL: https://www.kaggle.com/datasets/ronikdedhia/next-word-prediction
License(s): unknown
  0% 0.00/228k [00:00<?, ?B/s]
100% 228k/228k [00:00<00:00, 651MB/s]


In [4]:
import zipfile
zip_ref = zipfile.ZipFile('/content/next-word-prediction.zip', 'r')
zip_ref.extractall('/content')
zip_ref.close()

In [5]:
file_path = "/content/1661-0.txt"

with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

print(len(text))
print(text[:300])


581888
﻿
Project Gutenberg's The Adventures of Sherlock Holmes, by Arthur Conan Doyle

This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it away or
re-use it under the terms of the Project Gutenberg License included
with this eBook or


In [6]:
# remove unnecessary line break
text = text.replace("\n", " ")
text = text.replace("\r", " ")

In [7]:
# remove extra space
import re

text = re.sub(r"\s+", " ", text)
text = text.strip()


In [8]:
print(text[:500])


﻿ Project Gutenberg's The Adventures of Sherlock Holmes, by Arthur Conan Doyle This eBook is for the use of anyone anywhere at no cost and with almost no restrictions whatsoever. You may copy it, give it away or re-use it under the terms of the Project Gutenberg License included with this eBook or online at www.gutenberg.net Title: The Adventures of Sherlock Holmes Author: Arthur Conan Doyle Release Date: November 29, 2002 [EBook #1661] Last Updated: May 20, 2019 Language: English Character set 


In [9]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer

In [10]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])


In [11]:
len(tokenizer.word_index)

8931

In [12]:
input_sequences = []

tokenized_sequence = tokenizer.texts_to_sequences([text])[0]
sequence_length = 20
for i in range(sequence_length, len(tokenized_sequence)):
  seq = tokenized_sequence[i-sequence_length:i+1]
  input_sequences.append(seq)

input_sequences = np.array(input_sequences)
print(input_sequences.shape)

(111233, 21)


In [13]:
input_sequences

array([[4789,  145, 4790, ...,    4,  394, 2237],
       [ 145, 4790,    1, ...,  394, 2237,   21],
       [4790,    1, 1020, ..., 2237,   21,   51],
       ...,
       [ 884,  939,  552, ...,    3,  360,   83],
       [ 939,  552,  112, ...,  360,   83,  358],
       [ 552,  112,    3, ...,   83,  358, 1673]])

In [14]:
max_len = max([len(x) for x in input_sequences])
max_len

21

In [15]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

In [16]:
X.shape

(111233, 20)

In [17]:
y.shape

(111233,)

In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Input, Dense, Dropout

In [19]:
model = Sequential()
model.add(Input(shape = (max_len-1,)))
model.add(Embedding(len(tokenizer.word_index)+1, 100))
model.add(LSTM(150, return_sequences = True))
model.add(LSTM(150))
model.add(Dropout(.1))
model.add(Dense(len(tokenizer.word_index)+1, activation  = 'softmax'))

In [20]:
model.compile(loss = 'sparse_categorical_crossentropy', optimizer = 'adam', metrics = ['accuracy'])

In [21]:
history  = model.fit(X,y, epochs = 100, batch_size = 64)

Epoch 1/100
1739/1739 ━━━━━━━━━━━━━━━━━━━━ 22s 9ms/step - accuracy: 0.0513 - loss: 6.8813
Epoch 2/100
1739/1739 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - accuracy: 0.0668 - loss: 6.1898
Epoch 3/100
1739/1739 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - accuracy: 0.0787 - loss: 5.9480
Epoch 4/100
1739/1739 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - accuracy: 0.0996 - loss: 5.7054
Epoch 5/100
1739/1739 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - accuracy: 0.1138 - loss: 5.4718
Epoch 6/100
1739/1739 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - accuracy: 0.1236 - loss: 5.2906
Epoch 7/100
1739/1739 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - accuracy: 0.1309 - loss: 5.1580
Epoch 8/100
1739/1739 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - accuracy: 0.1381 - loss: 5.1370
Epoch 9/100
1739/1739 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - accuracy: 0.1465 - loss: 4.9316
Epoch 10/100
1739/1739 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - accuracy: 0.1518 - loss: 4.7673
Epoch 11/100
1739/1739 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - accuracy: 0.1592 - loss: 4.6117
Epoch 1

In [22]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
text_ = "what's your name"
max_len = 20
for i in range(50):
  token_text_ = tokenizer.texts_to_sequences([text_])[0]
  padded_token = pad_sequences([token_text_], maxlen=max_len, padding='pre')
  pos = np.argmax(model.predict(padded_token, verbose=0))
  for word, index in tokenizer.word_index.items():
    if index == pos:
      text_ += " " + word
      break
print(text_)

what's your name as to the awful consequences to have been as good as it is not very much to say that my hair were more than once more to the altar of the day when she came from her room and had a singular man who sat on the gloom and there


In [23]:
import pickle
model.save('sherlock.keras')

with open('tokenizer.pickle', 'wb') as handle:
  pickle.dump(tokenizer, handle, protocol = pickle.HIGHEST_PROTOCOL)

print('Model and Tokenizer saved! Download these two files')

Model and Tokenizer saved! Download these two files
